# SLC46A1 engraftment, step 5: chicken 144 by pairs, and the combined construct

Two things the earlier steps leave open.

Chicken 144 is the one high-risk humanising substitution with no coupling
readout, since the EVcouplings run does not fit that column, so the candidate
set for a double-substitution scan there is taken from the deposited
coordinates instead: the ten positions nearest it in 7BC6 and 7BC7, measured by
minimum heavy-atom distance. Part A scores every pair at those positions, the
same machinery step 4 used at chicken 407.

Every rescue so far has been measured one target at a time, against the
wild-type chicken background. Part B writes all twenty primary humanising
substitutions into one sequence, adds the chosen rescuers, and rescores each
primary with the others already in place. Where a rescuer and a primary want
different residues at the same position, both resolutions are scored and
neither is chosen here.

It stops on fatal conditions instead of warning, checkpoints to Drive after
every batch, and resumes from the checkpoint if the runtime is lost. Run every
cell in order.


In [ ]:
import os, sys, json, hashlib, datetime, time

from google.colab import drive
drive.mount('/content/drive')

PROJECT = '/content/drive/MyDrive/MIT/All Project Files/SLC46A1 engraftment'
OUT     = os.path.join(PROJECT, 'esm2_step5')

if not os.path.isdir(PROJECT):
    raise SystemExit('FATAL project folder not found: %s' % PROJECT)
os.makedirs(OUT, exist_ok=True)

REQUIRED = ['data/structural_candidates.json',
            'data/double_rescue_report.json',
            'data/rescue_sweep_report.json',
            'data/pharmacophore_position_set.tsv',
            'data/primary_substitution_risk.tsv',
            'data/esm2_step2/rescue_scan.tsv',
            'structures/7BC6_to_human.json']
missing = [f for f in REQUIRED if not os.path.exists(os.path.join(PROJECT, f))]
if missing:
    raise SystemExit('FATAL missing inputs in %s: %s' % (PROJECT, missing))

RUNLOG = os.path.join(OUT, 'run.log')

def log(msg):
    line = '%s  %s' % (datetime.datetime.utcnow().isoformat(timespec='seconds'), msg)
    with open(RUNLOG, 'a') as fh:
        fh.write(line + chr(10))
    print(line, flush=True)

log('start, project=%s' % PROJECT)


In [ ]:
import shutil, subprocess

if shutil.which('nvidia-smi') is None:
    raise SystemExit('FATAL this runtime has no GPU. Set Runtime > Change runtime '
                     'type > T4 GPU (or better), then run from the first cell again.')
if subprocess.run(['nvidia-smi'], capture_output=True).returncode != 0:
    raise SystemExit('FATAL nvidia-smi is present but failing, so the runtime has no '
                     'usable GPU. Change the runtime type and run from the first cell.')

subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'torch', 'transformers'],
               check=True)

import torch
from transformers import AutoTokenizer, EsmForMaskedLM

if not torch.cuda.is_available():
    raise SystemExit('FATAL torch cannot see the GPU after install')
log('torch %s, device %s' % (torch.__version__, torch.cuda.get_device_name(0)))


In [ ]:
import urllib.request

SEQ = os.path.join(OUT, 'seq')
os.makedirs(SEQ, exist_ok=True)

def fetch(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    urllib.request.urlretrieve(url, path)
    if os.path.getsize(path) == 0:
        raise SystemExit('FATAL empty download: %s' % url)
    return path

fetch('https://rest.uniprot.org/uniprotkb/Q96NT5.fasta', os.path.join(SEQ, 'Q96NT5.fasta'))
fetch('https://www.rcsb.org/fasta/entry/7BC6', os.path.join(SEQ, '7BC6.fasta'))

human = ''.join(l.strip() for l in open(os.path.join(SEQ, 'Q96NT5.fasta'))
                if not l.startswith('>'))
ent = {}
for rec in open(os.path.join(SEQ, '7BC6.fasta')).read().split('>'):
    if rec.strip():
        parts = rec.strip().split(chr(10))
        ent[parts[0].split('|')[0]] = ''.join(parts[1:])
chicken = ent['7BC6_1'][:473]

if len(human) != 459:
    raise SystemExit('FATAL Q96NT5 length %d, expected 459' % len(human))
if len(chicken) != 473:
    raise SystemExit('FATAL chicken native length %d, expected 473' % len(chicken))

CHICKEN_SHA = hashlib.sha256(chicken.encode()).hexdigest()
EXPECT_SHA = '5d68e7c91dc3af3cd898a3f459617f3b22670f3dbdf9f798e8d740d8669a8aea'
if CHICKEN_SHA != EXPECT_SHA:
    raise SystemExit('FATAL the chicken sequence differs from the one step 4 scored: '
                     '%s against %s' % (CHICKEN_SHA, EXPECT_SHA))
log('sequences ok: human %d, chicken %d, chicken_sha256 matches step 4'
    % (len(human), len(chicken)))

AA = 'ACDEFGHIKLMNPQRSTVWY'
THRESHOLD = 1e-3

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'pharmacophore_position_set.tsv'))]
pcol = {k: i for i, k in enumerate(rows[0])}
PSET = []
for r in rows[1:]:
    cp, hp = int(r[pcol['chicken_pos']]), int(r[pcol['human_pos']])
    if chicken[cp - 1] != r[pcol['chicken_aa']]:
        raise SystemExit('FATAL position set says chicken %d is %s, the sequence says %s'
                         % (cp, r[pcol['chicken_aa']], chicken[cp - 1]))
    if human[hp - 1] != r[pcol['human_aa']]:
        raise SystemExit('FATAL position set says human %d is %s, the sequence says %s'
                         % (hp, r[pcol['human_aa']], human[hp - 1]))
    PSET.append({'chicken_pos': cp, 'chicken_aa': r[pcol['chicken_aa']],
                 'human_pos': hp, 'human_aa': r[pcol['human_aa']],
                 'differs': r[pcol['differs']] == '1',
                 'parker': r[pcol['in_parker_named']] == '1',
                 'role': r[pcol['role_or_note']]})
PRIM = [p for p in PSET if p['differs']]
log('position set %d positions, %d primary humanising substitutions'
    % (len(PSET), len(PRIM)))


In [ ]:
TARGET_A = 144
N_POS = 10

sc = json.load(open(os.path.join(PROJECT, 'data', 'structural_candidates.json')))
if str(TARGET_A) not in sc['targets']:
    raise SystemExit('FATAL chicken %d has no structural ranking' % TARGET_A)
ta = sc['targets'][str(TARGET_A)]
if not ta['modelled']:
    raise SystemExit('FATAL chicken %d is not modelled in both structures' % TARGET_A)
if len(ta['candidates']) < N_POS:
    raise SystemExit('FATAL chicken %d has %d ranked candidates, %d needed'
                     % (TARGET_A, len(ta['candidates']), N_POS))
CAND_A = [int(c['chicken_pos']) for c in ta['candidates'][:N_POS]]
if len(set(CAND_A)) != N_POS or TARGET_A in CAND_A:
    raise SystemExit('FATAL the candidate list for %d is malformed: %s' % (TARGET_A, CAND_A))

rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'primary_substitution_risk.tsv'))]
rcol = {k: i for i, k in enumerate(rows[0])}
tr = [r for r in rows[1:] if int(r[rcol['chicken_pos']]) == TARGET_A]
if len(tr) != 1:
    raise SystemExit('FATAL chicken %d appears %d times in the risk table'
                     % (TARGET_A, len(tr)))
tr = tr[0]
A_CK_AA, A_HU_AA = tr[rcol['chicken_aa']], tr[rcol['human_aa']]
A_HU_POS = int(tr[rcol['human_pos']])
A_BASELINE = float(tr[rcol['p_human_wt_background']])
if chicken[TARGET_A - 1] != A_CK_AA or human[A_HU_POS - 1] != A_HU_AA:
    raise SystemExit('FATAL the risk table disagrees with the sequences at chicken %d'
                     % TARGET_A)
if A_BASELINE >= THRESHOLD:
    raise SystemExit('FATAL baseline %.3e at chicken %d is not below the threshold'
                     % (A_BASELINE, TARGET_A))

pairs_A = [(CAND_A[a], CAND_A[b]) for a in range(N_POS) for b in range(a + 1, N_POS)]
doubles_A = [(i, x, j, y) for (i, j) in pairs_A
             for x in AA if x != chicken[i - 1]
             for y in AA if y != chicken[j - 1]]
singles_A = [(j, y) for j in CAND_A for y in AA if y != chicken[j - 1]]

log('part A target %s%d to %s at human %d, baseline %.3e'
    % (A_CK_AA, TARGET_A, A_HU_AA, A_HU_POS, A_BASELINE))
log('part A candidates %s, chosen by distance in the deposited coordinates' % CAND_A)
log('part A pairs %d, doubles %d, singles %d'
    % (len(pairs_A), len(doubles_A), len(singles_A)))


In [ ]:
MODEL = 'facebook/esm2_t33_650M_UR50D'
BATCH = 48
SCHEMA = 'v5-construct'

tok = AutoTokenizer.from_pretrained(MODEL)
model = EsmForMaskedLM.from_pretrained(MODEL).eval().cuda()

base_ids = tok(chicken, return_tensors='pt')['input_ids'][0]
offset = 1
AAID = {y: tok.convert_tokens_to_ids(y) for y in AA}
if any(v is None or v < 0 for v in AAID.values()):
    raise SystemExit('FATAL the tokenizer has no id for one of the amino acids')
for p in PSET:
    if int(base_ids[p['chicken_pos'] - 1 + offset]) != AAID[p['chicken_aa']]:
        raise SystemExit('FATAL the token at chicken %d is not its residue, so the '
                         'offset is wrong' % p['chicken_pos'])

READ_IDX_A = TARGET_A - 1 + offset
READ_TOK_A = AAID[A_HU_AA]


def score_to(variants, ckpt, header, key_of, apply_of, read_idx, read_tok):
    done = set()
    if os.path.exists(ckpt) and os.path.getsize(ckpt) > 0:
        first = open(ckpt).readline().rstrip(chr(10))
        if first != header:
            raise SystemExit('FATAL checkpoint %s was written by a different schema; '
                             'delete it and rerun' % ckpt)
        for line in open(ckpt):
            p = line.rstrip(chr(10)).split(chr(9))
            if p[0] == 'row':
                done.add(key_of(p))
        log('resuming %s, already scored %d' % (os.path.basename(ckpt), len(done)))
    todo = [v for v in variants if key_of(['row'] + [str(x) for x in v]) not in done]
    log('%s: %d to score of %d' % (os.path.basename(ckpt), len(todo), len(variants)))
    t0 = time.time()
    with open(ckpt, 'a') as fh:
        if not done:
            fh.write(header + chr(10))
            fh.flush()
        for b in range(0, len(todo), BATCH):
            chunk = todo[b:b + BATCH]
            batch = base_ids.repeat(len(chunk), 1)
            for k, v in enumerate(chunk):
                apply_of(batch, k, v)
            with torch.no_grad():
                logits = model(batch.cuda()).logits
            sel = torch.softmax(logits[:, read_idx, :].float(), dim=-1)
            for k, v in enumerate(chunk):
                fh.write(chr(9).join(['row'] + [str(x) for x in v]
                                     + ['%.6e' % float(sel[k, read_tok])]) + chr(10))
            fh.flush()
            if (b // BATCH) % 25 == 0:
                log('  %d of %d, %.1f min elapsed'
                    % (b + len(chunk), len(todo), (time.time() - t0) / 60))
    log('%s complete, %.1f min' % (os.path.basename(ckpt), (time.time() - t0) / 60))


def apply_double(batch, k, v):
    i, x, j, y = v
    batch[k, i - 1 + offset] = AAID[x]
    batch[k, j - 1 + offset] = AAID[y]


def apply_single(batch, k, v):
    j, y = v
    batch[k, j - 1 + offset] = AAID[y]


SING_A = os.path.join(OUT, 'a144_singles_partial.tsv')
DBL_A = os.path.join(OUT, 'a144_doubles_partial.tsv')
HS = chr(9).join(['schema_' + SCHEMA, 'pos', 'aa', 'p_target'])
HD = chr(9).join(['schema_' + SCHEMA, 'pos_i', 'aa_i', 'pos_j', 'aa_j', 'p_target'])

score_to(singles_A, SING_A, HS, lambda p: (int(p[1]), p[2]), apply_single,
         READ_IDX_A, READ_TOK_A)
score_to(doubles_A, DBL_A, HD, lambda p: (int(p[1]), p[2], int(p[3]), p[4]),
         apply_double, READ_IDX_A, READ_TOK_A)


In [ ]:
singA = {}
for line in open(SING_A):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        singA[(int(p[1]), p[2])] = float(p[3])
dblA = {}
for line in open(DBL_A):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        dblA[(int(p[1]), p[2], int(p[3]), p[4])] = float(p[5])
if len(singA) != len(singles_A) or len(dblA) != len(doubles_A):
    raise SystemExit('FATAL part A checkpoints hold %d singles and %d doubles, %d and %d '
                     'expected' % (len(singA), len(dblA), len(singles_A), len(doubles_A)))

prevA = {}
scan = open(os.path.join(PROJECT, 'data', 'esm2_step2', 'rescue_scan.tsv'))
head = scan.readline().rstrip(chr(10)).split(chr(9))
scol = {k: i for i, k in enumerate(head)}
for line in scan:
    r = line.rstrip(chr(10)).split(chr(9))
    if int(r[scol['target_pos']]) != TARGET_A:
        continue
    prevA[(int(r[scol['rescuer_pos']]), r[scol['rescuer_to']])] = float(r[scol['p_target']])
if not prevA:
    raise SystemExit('FATAL the step 2 scan holds no rows for target %d' % TARGET_A)
TOL = 5e-2
worstA = (0.0, None)
for k, v in singA.items():
    if k not in prevA:
        raise SystemExit('FATAL %s%d is not in the step 2 scan' % (k[1], k[0]))
    rel = abs(v - prevA[k]) / max(v, prevA[k], 1e-30)
    if rel > worstA[0]:
        worstA = (rel, (k, v, prevA[k]))
if worstA[0] > TOL:
    k, a, b = worstA[1]
    raise SystemExit('FATAL the rescored singles disagree with step 2: %s%d gives %.6e '
                     'here and %.6e there, %.2e above the tolerance of %.0e'
                     % (k[1], k[0], a, b, worstA[0], TOL))
log('part A singles rechecked against step 2: %d compared, worst relative difference %.2e'
    % (len(singA), worstA[0]))

bestA_single = max(singA.items(), key=lambda kv: kv[1])
rowsA = []
for (i, x, j, y), p in dblA.items():
    pi, pj = singA[(i, x)], singA[(j, y)]
    rowsA.append({'pos_i': i, 'from_i': chicken[i - 1], 'to_i': x,
                  'pos_j': j, 'from_j': chicken[j - 1], 'to_j': y,
                  'p_target': p, 'p_single_i': pi, 'p_single_j': pj,
                  'best_single_of_pair': max(pi, pj),
                  'fold_vs_baseline': p / A_BASELINE,
                  'fold_vs_best_single_of_pair': p / max(pi, pj),
                  'crosses': p >= THRESHOLD})
rowsA.sort(key=lambda r: -r['p_target'])
colsA = ['pos_i', 'from_i', 'to_i', 'pos_j', 'from_j', 'to_j', 'p_target',
         'p_single_i', 'p_single_j', 'best_single_of_pair', 'fold_vs_baseline',
         'fold_vs_best_single_of_pair', 'crosses']
with open(os.path.join(OUT, 'a144_double_rescue.tsv'), 'w') as fh:
    fh.write(chr(9).join(colsA) + chr(10))
    for r in rowsA:
        fh.write(chr(9).join(('%.6e' % r[c]) if isinstance(r[c], float) else str(r[c])
                             for c in colsA) + chr(10))
with open(os.path.join(OUT, 'a144_singles_recheck.tsv'), 'w') as fh:
    fh.write(chr(9).join(['pos', 'from', 'to', 'p_target', 'p_target_step2']) + chr(10))
    for (j, y), v in sorted(singA.items()):
        fh.write(chr(9).join([str(j), chicken[j - 1], y, '%.6e' % v,
                              '%.6e' % prevA[(j, y)]]) + chr(10))

crossA = [r for r in rowsA if r['crosses']]
log('part A doubles crossing %.0e: %d of %d' % (THRESHOLD, len(crossA), len(rowsA)))
log('part A best single %s%d at %.6e' % (bestA_single[0][1], bestA_single[0][0],
                                         bestA_single[1]))
r = rowsA[0]
log('part A best double %s%d%s with %s%d%s at %.6e, %.1f-fold over baseline, %.2f-fold '
    'over the better half'
    % (r['from_i'], r['pos_i'], r['to_i'], r['from_j'], r['pos_j'], r['to_j'],
       r['p_target'], r['fold_vs_baseline'], r['fold_vs_best_single_of_pair']))


In [ ]:
sweep = json.load(open(os.path.join(PROJECT, 'data', 'rescue_sweep_report.json')))
dbl407 = json.load(open(os.path.join(PROJECT, 'data', 'double_rescue_report.json')))
RADIUS = sweep.get('published_radius', 3)

RESCUERS = []
for t, blocks in sweep['per_target'].items():
    t = int(t)
    pub = blocks.get('published', [])
    ok = [r for r in pub if r['crosses'] and r['separation'] <= RADIUS]
    if not ok:
        continue
    b = max(ok, key=lambda r: r['p_target'])
    RESCUERS.append({'target': t, 'pos': int(b['rescuer_pos']), 'to': b['rescuer_to'],
                     'p_target': b['p_target'], 'source': 'single, published policy'})

bd = dbl407['best_double']
for pos, to in ((bd['pos_i'], bd['to_i']), (bd['pos_j'], bd['to_j'])):
    RESCUERS.append({'target': dbl407['target_chicken_pos'], 'pos': int(pos), 'to': to,
                     'p_target': bd['p_target'], 'source': 'double, step 4'})

if crossA:
    ba = crossA[0]
    for pos, to in ((ba['pos_i'], ba['to_i']), (ba['pos_j'], ba['to_j'])):
        RESCUERS.append({'target': TARGET_A, 'pos': int(pos), 'to': to,
                         'p_target': ba['p_target'], 'source': 'double, part A'})
    log('part A rescued chicken %d, so its pair joins the construct' % TARGET_A)
else:
    log('part A did not rescue chicken %d, so it carries no rescuer in the construct'
        % TARGET_A)

PRIM_AT = {p['chicken_pos']: p for p in PRIM}
SET_AT = {p['chicken_pos']: p for p in PSET}
collisions = []
for r in RESCUERS:
    p = SET_AT.get(r['pos'])
    if p is None:
        r['collision'] = 'none'
        continue
    if p['differs']:
        r['collision'] = 'primary'
        collisions.append(r)
    elif r['to'] != p['chicken_aa']:
        r['collision'] = 'conserved position in the set'
        collisions.append(r)
    else:
        r['collision'] = 'none'

seen = {}
for r in RESCUERS:
    k = (r['pos'], r['to'])
    seen.setdefault(k, []).append(r['target'])
dups = {k: v for k, v in seen.items() if len(v) > 1}
clash = {}
for r in RESCUERS:
    clash.setdefault(r['pos'], set()).add(r['to'])
multi = {p: s for p, s in clash.items() if len(s) > 1}
if multi:
    raise SystemExit('FATAL two rescuers want different residues at the same position: %s'
                     % multi)

log('')
log('rescuers chosen')
log('  target  position  to  collision                      source')
for r in sorted(RESCUERS, key=lambda r: r['target']):
    log('  %6d %9d %3s  %-29s %s'
        % (r['target'], r['pos'], r['to'], r['collision'], r['source']))
log('collisions with the position set: %d' % len(collisions))
for r in collisions:
    p = SET_AT[r['pos']]
    log('  chicken %d: the rescue for %d wants %s, the set has chicken %s and human %s%s'
        % (r['pos'], r['target'], r['to'], p['chicken_aa'], p['human_aa'],
           ', a primary humanising substitution' if p['differs'] else
           ', conserved between the species'))


def build(primaries_in, rescuers_in, leave_out=None, conflict='human'):
    s = list(chicken)
    for p in PRIM:
        if not primaries_in or p['chicken_pos'] == leave_out:
            continue
        s[p['chicken_pos'] - 1] = p['human_aa']
    if rescuers_in:
        for r in RESCUERS:
            if r['pos'] == leave_out:
                continue
            p = SET_AT.get(r['pos'])
            if (primaries_in and p is not None and p['differs']
                    and conflict == 'human'):
                continue
            s[r['pos'] - 1] = r['to']
    return ''.join(s)


BACKGROUNDS = [('wt', False, False, 'human'),
               ('primaries', True, False, 'human'),
               ('primaries_rescuers_primary_wins', True, True, 'human'),
               ('primaries_rescuers_rescue_wins', True, True, 'rescue')]
log('')
for name, pi, ri, cf in BACKGROUNDS:
    s = build(pi, ri, None, cf)
    diff = sum(1 for a, b in zip(s, chicken) if a != b)
    log('background %-34s differs from chicken at %d positions' % (name, diff))


In [ ]:
READS = [(p['chicken_pos'], p['human_aa']) for p in PRIM]
jobs = []
for name, pi, ri, cf in BACKGROUNDS:
    for cp, ha in READS:
        jobs.append((name, pi, ri, cf, cp, ha))

CK5 = os.path.join(OUT, 'construct_partial.tsv')
H5 = chr(9).join(['schema_' + SCHEMA, 'background', 'chicken_pos', 'human_aa',
                  'p_human'])
done = set()
if os.path.exists(CK5) and os.path.getsize(CK5) > 0:
    first = open(CK5).readline().rstrip(chr(10))
    if first != H5:
        raise SystemExit('FATAL checkpoint %s was written by a different schema; delete '
                         'it and rerun' % CK5)
    for line in open(CK5):
        p = line.rstrip(chr(10)).split(chr(9))
        if p[0] == 'row':
            done.add((p[1], int(p[2])))
    log('resuming the construct scoring, already done %d' % len(done))

todo = [j for j in jobs if (j[0], j[4]) not in done]
log('construct scoring: %d of %d to do' % (len(todo), len(jobs)))
t0 = time.time()
with open(CK5, 'a') as fh:
    if not done:
        fh.write(H5 + chr(10))
        fh.flush()
    for b in range(0, len(todo), BATCH):
        chunk = todo[b:b + BATCH]
        seqs = [build(pi, ri, cp, cf) for (_, pi, ri, cf, cp, _) in chunk]
        enc = tok(seqs, return_tensors='pt')['input_ids']
        if enc.shape[1] != base_ids.shape[0]:
            raise SystemExit('FATAL a constructed sequence tokenizes to %d tokens, the '
                             'chicken sequence to %d' % (enc.shape[1], base_ids.shape[0]))
        with torch.no_grad():
            logits = model(enc.cuda()).logits
        probs = torch.softmax(logits.float(), dim=-1)
        for k, (name, pi, ri, cf, cp, ha) in enumerate(chunk):
            idx = cp - 1 + offset
            if int(enc[k, idx]) != AAID[chicken[cp - 1]]:
                raise SystemExit('FATAL the left-out position %d does not carry the '
                                 'chicken residue in background %s' % (cp, name))
            fh.write(chr(9).join(['row', name, str(cp), ha,
                                  '%.6e' % float(probs[k, idx, AAID[ha]])]) + chr(10))
        fh.flush()
log('construct scoring complete, %.1f min' % ((time.time() - t0) / 60))


In [ ]:
P = {}
for line in open(CK5):
    p = line.rstrip(chr(10)).split(chr(9))
    if p[0] == 'row':
        P[(p[1], int(p[2]))] = float(p[4])
if len(P) != len(jobs):
    raise SystemExit('FATAL the construct checkpoint holds %d rows, %d expected'
                     % (len(P), len(jobs)))

risk = {}
rows = [l.rstrip(chr(10)).split(chr(9))
        for l in open(os.path.join(PROJECT, 'data', 'primary_substitution_risk.tsv'))]
rc = {k: i for i, k in enumerate(rows[0])}
for r in rows[1:]:
    risk[int(r[rc['chicken_pos']])] = float(r[rc['p_human_wt_background']])

worst = (0.0, None)
for cp, _ in READS:
    if cp not in risk:
        raise SystemExit('FATAL chicken %d is not in the step 1 risk table' % cp)
    a, b = P[('wt', cp)], risk[cp]
    rel = abs(a - b) / max(a, b, 1e-30)
    if rel > worst[0]:
        worst = (rel, (cp, a, b))
if worst[0] > TOL:
    cp, a, b = worst[1]
    raise SystemExit('FATAL the wild-type background does not reproduce step 1 at chicken '
                     '%d: %.6e here and %.6e there, %.2e above the tolerance of %.0e'
                     % (cp, a, b, worst[0], TOL))
log('wild-type background reproduces step 1 at all %d primaries, worst relative '
    'difference %.2e' % (len(READS), worst[0]))

names = [b[0] for b in BACKGROUNDS]
out_rows = []
for cp, ha in sorted(READS):
    rec = {'chicken_pos': cp, 'human_aa': ha,
           'p_baseline_step1': risk[cp]}
    for n in names:
        rec[n] = P[(n, cp)]
    rec['fold_primaries_over_wt'] = P[('primaries', cp)] / P[('wt', cp)]
    rec['fold_final_over_wt'] = P[('primaries_rescuers_primary_wins', cp)] / P[('wt', cp)]
    out_rows.append(rec)

cols = (['chicken_pos', 'human_aa', 'p_baseline_step1'] + names
        + ['fold_primaries_over_wt', 'fold_final_over_wt'])
with open(os.path.join(OUT, 'construct_profile.tsv'), 'w') as fh:
    fh.write(chr(9).join(cols) + chr(10))
    for r in out_rows:
        fh.write(chr(9).join(('%.6e' % r[c]) if isinstance(r[c], float) else str(r[c])
                             for c in cols) + chr(10))

log('')
log('each primary scored with the others already written in')
log('chicken  human   wild type   primaries  primary wins  rescue wins  fold, final '
    'over wild type')
for r in out_rows:
    log('%7d %6s %11.3e %11.3e %13.3e %12.3e %14.2f'
        % (r['chicken_pos'], r['human_aa'], r['wt'], r['primaries'],
           r['primaries_rescuers_primary_wins'], r['primaries_rescuers_rescue_wins'],
           r['fold_final_over_wt']))

for n in names:
    below = [r for r in out_rows if r[n] < THRESHOLD]
    log('background %-34s below %.0e at %d of %d primaries'
        % (n, THRESHOLD, len(below), len(out_rows)))
    if below:
        log('   %s' % ', '.join('%d' % r['chicken_pos'] for r in below))

summary = {'schema': SCHEMA, 'model': MODEL, 'chicken_sha256': CHICKEN_SHA,
           'threshold': THRESHOLD, 'n_primaries': len(PRIM),
           'part_a': {'target': TARGET_A, 'candidates': CAND_A,
                      'candidate_source': 'minimum heavy-atom distance in 7BC6 and 7BC7',
                      'n_doubles': len(rowsA), 'n_singles': len(singA),
                      'worst_relative_difference_vs_step2': worstA[0],
                      'n_crossing': len(crossA),
                      'best_single': {'pos': bestA_single[0][0],
                                      'to': bestA_single[0][1], 'p': bestA_single[1]},
                      'best_double': rowsA[0]},
           'rescuers': RESCUERS,
           'collisions': [{'pos': r['pos'], 'to': r['to'], 'target': r['target'],
                           'kind': r['collision']} for r in collisions],
           'backgrounds': names,
           'wt_vs_step1_worst_relative_difference': worst[0],
           'below_threshold': {n: [r['chicken_pos'] for r in out_rows if r[n] < THRESHOLD]
                               for n in names},
           'profile': out_rows}
with open(os.path.join(OUT, 'construct_summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1)

log('')
log('wrote a144_double_rescue.tsv, a144_singles_recheck.tsv, construct_profile.tsv and '
    'construct_summary.json')
log('copy esm2_step5/ from Drive into data/ and run codes/18_construct_read.py')
